## 01 — Dataset + QLoRA fine-tuning (Colab)

This notebook:

- Generates a **260-example** synthetic dataset (train/valid/test) as JSONL
- Fine-tunes a **<1B** model with **QLoRA** (Unsloth)
- Uses **early stopping** on validation loss
- Saves LoRA adapters and training logs

> Recommended runtime: **Colab GPU (T4)**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab setup (GPU required for Unsloth)
import os, sys
os.environ["WANDB_DISABLED"] = "true"

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU detected. In Colab: Runtime → Change runtime type → Hardware accelerator: GPU. "
        "Then Runtime → Restart session, and re-run this cell."
    )

# If you previously installed CPU-only torch, reinstall the CUDA build.
# Colab typically supports cu121 wheels.
!pip -q install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install training/eval stack (avoid breaking Colab with pandas/protobuf majors)
!pip -q install -U "unsloth[colab]" transformers datasets accelerate peft trl bitsandbytes evaluate sacrebleu rouge-score scikit-learn matplotlib seaborn "pandas<3" "protobuf>=4.25.3,<6"

In [ ]:
# Repo bootstrap (prevents missing data/test.jsonl issues)
# If you opened this notebook from Drive, this cell ensures we run inside the cloned repo.

import os
from pathlib import Path

repo_dir = Path("/content/tatweer_challenge")
if not repo_dir.exists():
    !git clone -q https://github.com/Shamem-cyberx/tatweer_challenge.git /content/tatweer_challenge

os.chdir(str(repo_dir / "notebooks"))
print("cwd:", os.getcwd())
print("repo contents:", list(repo_dir.iterdir())[:5])

In [ ]:
# Generate dataset (writes JSONL to ../data/processed)
import json, random
from pathlib import Path

root = Path("..").resolve()  # your-name-challenge/
out_dir = root / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)


def ps(code: str) -> str:
    return f"```powershell\n{code.strip()}\n```"


def join_steps(steps):
    return "\n".join([f"{i+1}. {s}" for i, s in enumerate(steps)])


def build_examples(seed: int = 3407):
    rng = random.Random(seed)
    examples = []

    # A compact in-notebook version of the generator (kept deterministic)
    examples.append({
        "id": "network_dns_cache",
        "category": "network",
        "expects_code": True,
        "difficulty": 1,
        "instruction": "My Windows PC can open some websites but others fail with DNS errors. Give me PowerShell steps to diagnose and fix DNS issues safely.",
        "response": "\n".join([
            "Start with read-only checks, then reset DNS client state if needed.",
            "",
            join_steps([
                "Check current DNS server configuration and adapter status.",
                "Test name resolution against your configured DNS and a public resolver.",
                "Flush the DNS client cache (safe) and retry.",
                "If it still fails, reset the DNS client service and renew the IP lease.",
            ]),
            "",
            ps("""
Get-NetAdapter | Sort-Object Status -Descending | Format-Table -Auto Name, Status, LinkSpeed
Get-DnsClientServerAddress -AddressFamily IPv4 | Format-Table -Auto InterfaceAlias, ServerAddresses

Resolve-DnsName -Name "www.microsoft.com" -ErrorAction SilentlyContinue
Resolve-DnsName -Name "www.microsoft.com" -Server 1.1.1.1 -ErrorAction SilentlyContinue

Clear-DnsClientCache

Restart-Service -Name Dnscache -Force
ipconfig /renew
"""),
            "",
            "If `Resolve-DnsName` works only with `-Server 1.1.1.1`, your local DNS server is likely the problem (router/ISP/enterprise DNS).",
        ])
    })

    proxy_hosts = ["login.company.tld", "intranet.local", "api.vendor.tld"]
    for i, h in enumerate(proxy_hosts):
        examples.append({
            "id": f"network_proxy_{i}",
            "category": "network",
            "expects_code": True,
            "difficulty": 2,
            "instruction": f"Ping is blocked on my network. I can’t reach {h}. How do I troubleshoot connectivity using PowerShell without ICMP?",
            "response": "\n".join([
                "If ICMP is blocked, validate DNS, route, and TCP connectivity instead of ping.",
                "",
                ps(f"""
$hostName = "{h}"
Resolve-DnsName -Name $hostName -ErrorAction SilentlyContinue

Test-NetConnection -ComputerName $hostName -Port 443 -InformationLevel Detailed
Test-NetConnection -ComputerName $hostName -Port 80  -InformationLevel Detailed

tracert -d $hostName
"""),
                "",
                "If DNS fails, fix name resolution first. If DNS works but TCP fails, check firewall/proxy requirements and local outbound rules.",
            ])
        })

    services = [("Spooler", "Print Spooler"), ("wuauserv", "Windows Update"), ("WinDefend", "Microsoft Defender Antivirus Service")]
    for svc, desc in services:
        examples.append({
            "id": f"svc_restart_{svc.lower()}",
            "category": "services",
            "expects_code": True,
            "difficulty": 1,
            "instruction": f"{desc} seems stuck. How do I check its status and restart it in PowerShell?",
            "response": "\n".join([
                "Check status and recent dependent services first, then restart. If you see access denied, run PowerShell as Administrator.",
                "",
                ps(f"""
$name = "{svc}"
Get-Service -Name $name | Format-List *

(Get-Service -Name $name).DependentServices | Format-Table -Auto Name, Status

Restart-Service -Name $name -Force
Get-Service -Name $name
"""),
            ])
        })

    # Pad to 260 with templated “explain error” items
    error_templates = [
        ("network", "Name resolution failure", "Resolve-DnsName"),
        ("services", "Service cannot be started", "Get-WinEvent"),
        ("permissions", "Access is denied", "Get-Acl"),
        ("disk", "The disk is write-protected", "Get-Volume"),
        ("processes", "Process is not responding", "Get-Process"),
    ]

    idx = 0
    while len(examples) < 260:
        cat, err, cmd = error_templates[idx % len(error_templates)]
        idx += 1
        examples.append({
            "id": f"err_explain_{idx:03d}",
            "category": cat,
            "expects_code": True,
            "difficulty": 2 if (idx % 3) else 3,
            "instruction": f"I'm seeing this error in PowerShell: '{err}'. Explain likely causes and give a short diagnostic checklist with commands.",
            "response": "\n".join([
                "That error usually points to a few common root causes. Start with quick diagnostics before changing configuration.",
                "",
                join_steps([
                    "Confirm the exact command and parameters that triggered the error.",
                    "Check whether you have the required privileges (standard user vs admin).",
                    "Collect the most relevant state using built-in commands.",
                    "If the issue is reproducible, capture an event log snippet around the failure time.",
                ]),
                "",
                ps(f"""
whoami
{cmd} | Out-String | Select-Object -First 1
"""),
                "",
                "If you paste the exact command and the full error (including any category info), the next steps can be narrowed down safely.",
            ])
        })

    rng.shuffle(examples)
    return examples


examples = build_examples(seed=3407)
assert len(examples) >= 260

train, valid, test = examples[:220], examples[220:240], examples[240:260]

for name, split in [("train", train), ("valid", valid), ("test", test)]:
    path = out_dir / f"{name}.jsonl"
    with path.open("w", encoding="utf-8") as f:
        for ex in split:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Wrote:")
for name in ["train", "valid", "test"]:
    print(name, sum(1 for _ in (out_dir / f"{name}.jsonl").open("r", encoding="utf-8")))

In [ ]:
# Load dataset
from datasets import load_dataset
from pathlib import Path

root = Path("..").resolve()
data_dir = root / "data" / "processed"

raw = load_dataset(
    "json",
    data_files={
        "train": str(data_dir / "train.jsonl"),
        "validation": str(data_dir / "valid.jsonl"),
        "test": str(data_dir / "test.jsonl"),
    },
)
raw

In [ ]:
# Convert to SFT text format (instruction-response)

def to_sft_text(ex):
    return {
        "text": "\n".join([
            "### Instruction:",
            ex["instruction"].strip(),
            "",
            "### Response:",
            ex["response"].strip(),
        ])
    }

train_ds = raw["train"].map(to_sft_text, remove_columns=raw["train"].column_names)
val_ds = raw["validation"].map(to_sft_text, remove_columns=raw["validation"].column_names)

train_ds[0]["text"][:600]

In [ ]:
# QLoRA fine-tuning (Unsloth)
import torch
from unsloth import FastLanguageModel

model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # < 1B params
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# LoRA config (justify in your report)
# - r=16: enough capacity for domain adaptation on small dataset
# - alpha=32: stable scaling
# - dropout=0.05: helps prevent overfitting on 220 train samples
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

out_root = Path("..").resolve() / "outputs"
out_root.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(out_root / "checkpoints"),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # effective batch = 16
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to=[],
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=args,
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=3))

train_result = trainer.train()
train_result

In [ ]:
# Save adapters + plot loss curves
import json
import matplotlib.pyplot as plt

save_dir = Path("..").resolve() / "outputs" / "lora_adapters"
save_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

# Extract loss curves
history = trainer.state.log_history
train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for row in history:
    if "loss" in row and "step" in row:
        train_steps.append(row["step"])
        train_loss.append(row["loss"])
    if "eval_loss" in row and "step" in row:
        eval_steps.append(row["step"])
        eval_loss.append(row["eval_loss"])

plt.figure(figsize=(8,4))
if train_steps:
    plt.plot(train_steps, train_loss, label="train loss")
if eval_steps:
    plt.plot(eval_steps, eval_loss, label="valid loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.tight_layout()

plot_path = Path("..").resolve() / "outputs" / "loss_curve.png"
plt.savefig(plot_path, dpi=160)
print("Saved:", plot_path)

with open(Path("..").resolve() / "outputs" / "trainer_log_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("Saved adapters to:", save_dir)